In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "model"
MODEL_DIR.mkdir(exist_ok=True)

train_path = DATA_DIR / "food_drug_pairs_train.csv"
train_df = pd.read_csv(train_path)

print("train_df:", train_df.shape)
train_df["severity_silver"].value_counts()


train_df: (7152, 24)


severity_silver
0    5000
1    2128
2      24
Name: count, dtype: int64

In [2]:
import sys
print("Python exe:", sys.executable)
print("Python version:", sys.version)

Python exe: C:\Users\User\AppData\Local\Programs\Python\Python312\python.exe
Python version: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]


In [3]:
import sys
!"{sys.executable}" -m pip install joblib scikit-learn --upgrade



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [4]:
import joblib
import sklearn
print("joblib:", joblib.__version__)
print("sklearn:", sklearn.__version__)


joblib: 1.5.3
sklearn: 1.8.0


In [5]:
from sklearn.ensemble import RandomForestClassifier
print("RandomForest import OK ✅")


RandomForest import OK ✅


In [6]:
# Feature columns
feature_cols = [
    "Chemical_Class","Habit_Forming","Therapeutic_Class","Action_Class",
    "energy","protein","fat","carbs","fiber",
    "calcium","iron","vitamin_c","vitamin_a","vitamin_k_proxy",
    "is_alcohol","is_leafy_green"
]

# Ensure all features exist
for c in feature_cols:
    if c not in train_df.columns:
        train_df[c] = 0

X = train_df[feature_cols].copy()
y = train_df["severity_silver"].astype(int)

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())


X shape: (7152, 16)
y distribution:
 severity_silver
0    5000
1    2128
2      24
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("y_train counts:\n", y_train.value_counts())
print("y_test counts:\n", y_test.value_counts())


Train: (5721, 16) Test: (1431, 16)
y_train counts:
 severity_silver
0    4000
1    1702
2      19
Name: count, dtype: int64
y_test counts:
 severity_silver
0    1000
1     426
2       5
Name: count, dtype: int64


In [8]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

clf.fit(X_train, y_train)

print("Model trained successfully ✅")


Model trained successfully ✅


In [9]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = clf.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))


Confusion Matrix:
[[990   9   1]
 [  8 418   0]
 [  0   0   5]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9920    0.9900    0.9910      1000
           1     0.9789    0.9812    0.9801       426
           2     0.8333    1.0000    0.9091         5

    accuracy                         0.9874      1431
   macro avg     0.9347    0.9904    0.9601      1431
weighted avg     0.9875    0.9874    0.9875      1431



In [10]:
# List of all possible reason tags
ALL_REASON_TAGS = [
    "cns_alcohol",
    "calcium_antibiotic",
    "high_fat_empty_stomach",
    "iron_levothyroxine",
    "vitk_warfarin"
]

# Create binary columns for each reason
for tag in ALL_REASON_TAGS:
    train_df[tag] = train_df["reason_tags_silver"].fillna("").str.contains(tag).astype(int)

# Check distribution
train_df[ALL_REASON_TAGS].sum()


cns_alcohol                 24
calcium_antibiotic         546
high_fat_empty_stomach    1582
iron_levothyroxine           0
vitk_warfarin                0
dtype: int64

In [11]:
from sklearn.multiclass import OneVsRestClassifier

X = train_df[feature_cols]
Y = train_df[ALL_REASON_TAGS]

X_train_r, X_test_r, Y_train_r, Y_test_r = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

reason_model = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
)

reason_model.fit(X_train_r, Y_train_r)

print("Reason model trained ✅")


Reason model trained ✅


C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 3 is present in all training examples.
  warnings.warn(
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(


In [12]:
from sklearn.metrics import classification_report

Y_pred_r = reason_model.predict(X_test_r)

for i, tag in enumerate(ALL_REASON_TAGS):
    print(f"\n===== Reason: {tag} =====")
    print(classification_report(
        Y_test_r.iloc[:, i],
        Y_pred_r[:, i],
        digits=4,
        zero_division=0
    ))



===== Reason: cns_alcohol =====
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      1427
           1     1.0000    1.0000    1.0000         4

    accuracy                         1.0000      1431
   macro avg     1.0000    1.0000    1.0000      1431
weighted avg     1.0000    1.0000    1.0000      1431


===== Reason: calcium_antibiotic =====
              precision    recall  f1-score   support

           0     1.0000    0.9947    0.9973      1317
           1     0.9421    1.0000    0.9702       114

    accuracy                         0.9951      1431
   macro avg     0.9711    0.9973    0.9838      1431
weighted avg     0.9954    0.9951    0.9952      1431


===== Reason: high_fat_empty_stomach =====
              precision    recall  f1-score   support

           0     0.9910    0.9821    0.9865      1120
           1     0.9377    0.9678    0.9525       311

    accuracy                         0.9790      1431
   macro av

In [20]:
import joblib

joblib.dump(reason_model, MODEL_DIR / "reason_model.pkl")
print("Saved reason_model.pkl ")


Saved reason_model.pkl 


In [13]:
import joblib
import json
from pathlib import Path

MODEL_DIR = Path("../model")
MODEL_DIR.mkdir(exist_ok=True)

# Save trained severity model
joblib.dump(clf, MODEL_DIR / "severity_model.pkl")

# Save feature column order
with open(MODEL_DIR / "severity_features.json", "w") as f:
    json.dump(feature_cols, f)

print("✅ Saved severity_model.pkl")
print("✅ Saved severity_features.json")


✅ Saved severity_model.pkl
✅ Saved severity_features.json
